# <span style="color: #1F1DB5;">SPRINT 4 – Manipulación de datos (Data Wrangling) Continuación

# <span style="color: #1F1DB5;">Notebook 1 – Manejo Avanzado de Datos

## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Aplicar operaciones avanzadas de **merge, join y concatenación** de DataFrames
- Usar **pivot tables** y **crosstabs** para resumir datos multidimensionales
- Trabajar con **datos de texto** usando métodos `str` de Pandas
- Aplicar **funciones personalizadas** con `apply()` y `map()`
- Comprender la diferencia entre **loc e iloc** con ejemplos visuales
- Aplicar técnicas básicas de **Feature Engineering**: binning, log transform

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Merge, join y concatenación | 25 min |
Pivot tables y crosstabs | 20 min |
Operaciones con texto (str methods) | 20 min |
apply() y Feature Engineering básico | 15 min |
Actividad práctica (Breakout Rooms) | 15 min |
Tips y buenas prácticas | 5 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo combinar y transformar datos de **múltiples fuentes**?

## <span style="color: #2826DE;">Merge, join y concatenación (25 mins)

Cuando trabajas con datos reales, raramente tienes toda la informacion en una sola tabla. Necesitas combinar datos de multiples fuentes. Vamos a trabajar con dos tablas: clientes y compras. Observa que no todos los clientes tienen compras y no todas las compras tienen un cliente registrado.

In [ ]:
import pandas as pd
import numpy as np

# Datasets de ejemplo
clientes = pd.DataFrame({
    'id_cliente': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Luis', 'María', 'Carlos', 'Elena'],
    'ciudad': ['México', 'España', 'Colombia', 'Argentina', 'Chile']
})

compras = pd.DataFrame({
    'id_cliente': [1, 2, 2, 3, 5, 6],
    'producto': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares', 'Webcam'],
    'precio': [1200, 25, 80, 350, 150, 90]
})

print('Clientes:')
print(clientes)
print('\nCompras:')
print(compras)

### Preguntas:

- ¿Cuantas filas tiene el INNER JOIN? Por que menos que la tabla de compras?

- ¿En el LEFT JOIN, que valor aparece en las columnas de compras para los clientes sin compras?

- ¿Cuando usarias un RIGHT JOIN o un FULL OUTER JOIN?

In [ ]:
# INNER JOIN: solo clientes con compras
inner = pd.merge(clientes, compras, on='id_cliente', how='inner')
print('INNER JOIN (solo coincidencias):')
print(inner)

# LEFT JOIN: todos los clientes, aunque no tengan compras
left = pd.merge(clientes, compras, on='id_cliente', how='left')
print('\nLEFT JOIN (todos los clientes):')
print(left)

# Concatenar DataFrames
df1 = clientes.iloc[:3]
df2 = clientes.iloc[3:]
combinado = pd.concat([df1, df2], ignore_index=True)
print('\nConcatenación:')
print(combinado)

### Preguntas:

- ¿Que diferencia hay entre `pd.merge()` y `pd.concat()`?

- ¿Cuando usarias `concat` en lugar de `merge`?

## <span style="color: #2826DE;">Pivot tables y crosstabs (20 mins)

Las pivot tables son una de las herramientas mas poderosas para resumir datos multidimensionales. En lugar de ver fila por fila, puedes ver patrones cruzando dos variables. Vamos a crear una tabla que muestre la masa promedio de cada especie de pinguino en cada isla.

In [ ]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
df = pd.read_csv(url).dropna()

# Pivot table: masa promedio por especie e isla
pivot = df.pivot_table(
    values='body_mass_g',
    index='species',
    columns='island',
    aggfunc='mean'
).round(0)

print('Masa promedio por especie e isla:')
print(pivot)

# Crosstab: conteo de pingüinos por especie y sexo
cross = pd.crosstab(df['species'], df['sex'])
print('\nConteo por especie y sexo:')
print(cross)

### Preguntas:

- ¿Que especie tiene la mayor masa promedio? En que isla?

- ¿Por que algunos valores de la pivot table son NaN?

- ¿Que diferencia hay entre `pivot_table` y `crosstab`?

## <span style="color: #2826DE;">Operaciones con texto – str methods (20 mins)

Los datos de texto son muy comunes en Data Science: nombres, emails, categorias, descripciones. Pandas tiene metodos `str` que te permiten limpiar y transformar texto de forma vectorizada, sin necesidad de bucles.

In [ ]:
df_texto = pd.DataFrame({
    'nombre': ['  Ana Garcia  ', 'LUIS MARTINEZ', 'maria lopez', 'Carlos-Ruiz'],
    'email': ['ana@gmail.com', 'luis@hotmail.com', 'maria@yahoo.com', 'carlos@empresa.com'],
    'telefono': ['555-1234', '555.5678', '555 9012', '5553456']
})

# Limpiar nombres
df_texto['nombre_limpio'] = df_texto['nombre'].str.strip().str.title()

# Extraer dominio del email
df_texto['dominio'] = df_texto['email'].str.split('@').str[1]

# Verificar si contiene gmail
df_texto['es_gmail'] = df_texto['email'].str.contains('gmail')

# Estandarizar teléfono
df_texto['tel_limpio'] = df_texto['telefono'].str.replace(r'[-. ]', '', regex=True)

print(df_texto[['nombre_limpio', 'dominio', 'es_gmail', 'tel_limpio']])

### Preguntas:

- ¿Por que es importante estandarizar el texto antes de analizar datos?

- ¿Que otros problemas de texto podrias encontrar en datos reales?

- ¿Como usarias `str.contains()` para filtrar emails de un dominio especifico?

## <span style="color: #2826DE;">apply() y Feature Engineering básico (15 mins)

La funcion `apply()` te permite aplicar cualquier funcion Python a cada elemento de una columna. Es mas flexible que los metodos vectorizados pero mas lento. Usala cuando necesitas logica compleja que no puedes expresar con operaciones simples.

In [ ]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
df = pd.read_csv(url).dropna()

# Función personalizada con apply()
def clasificar_pinguino(masa):
    if masa < 3500:
        return 'Pequeño'
    elif masa < 4500:
        return 'Mediano'
    else:
        return 'Grande'

df['tamaño'] = df['body_mass_g'].apply(clasificar_pinguino)

# Binning con pd.cut()
df['masa_categoria'] = pd.cut(
    df['body_mass_g'],
    bins=[0, 3500, 4500, 7000],
    labels=['Liviano', 'Medio', 'Pesado']
)

# Log transform
df['log_masa'] = np.log(df['body_mass_g'])

# Ratio entre variables
df['ratio_pico'] = df['bill_length_mm'] / df['bill_depth_mm']

print(df[['species', 'body_mass_g', 'tamaño', 'masa_categoria', 'ratio_pico']].head(8))

### Preguntas:

- ¿Por que el log transform reduce el sesgo en distribuciones asimetricas?

- ¿Que ventaja tiene `pd.cut()` sobre escribir multiples condiciones `if/elif`?

- ¿Cuando usarias `apply()` y cuando usarias operaciones vectorizadas como `df['col'] * 2`?

## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (15 mins)

### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

### Ejercicio

Usando el dataset de pingüinos:
1. Crea un DataFrame de islas con su latitud aproximada y haz un merge
2. Crea una pivot table con la masa promedio por especie y sexo
3. Crea una columna `ratio_pico` = bill_length_mm / bill_depth_mm
4. Clasifica los pingüinos en 3 categorías de tamaño usando binning
5. Aplica log transform a `body_mass_g` y compara la distribución

In [ ]:
# Tu código aquí


## <span style="color: #2826DE;">Tips y buenas prácticas

- **Prefiere merge sobre join** para mayor claridad
- **Usa pivot_table** cuando necesites resumir datos multidimensionales
- **Documenta las transformaciones** de feature engineering
- **Verifica el shape** antes y después de cada merge
- **loc para etiquetas, iloc para posiciones** — no los mezcles

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas


1️⃣ Diferencia entre loc e iloc?

- Son exactamente lo mismo
- loc es mas rapido que iloc
- iloc incluye el ultimo elemento, loc no
- **loc usa etiquetas, iloc usa posicion numerica** ✅

2️⃣ Que hace INNER JOIN?

- Devuelve todas las filas de ambas tablas
- Devuelve todas las filas de la tabla izquierda
- Devuelve todas las filas de la tabla derecha
- **Solo devuelve filas con coincidencias en ambas tablas** ✅

3️⃣ Para que sirve pivot_table?

- Para unir dos DataFrames
- Para limpiar valores nulos
- Para ordenar el DataFrame
- **Para resumir datos multidimensionales con agregaciones** ✅

4️⃣ Que hace str.contains()?

- Cuenta cuantas veces aparece un patron
- Reemplaza un patron por otro
- Divide el string en una lista
- **Verifica si un string contiene un patron** ✅

5️⃣ Para que sirve el log transform?

- Para normalizar entre 0 y 1
- Para convertir categorias en numeros
- Para eliminar outliers
- **Para reducir el sesgo en distribuciones asimetricas** ✅

6️⃣ Que hace pd.cut()?

- Elimina filas duplicadas
- Divide el DataFrame en dos partes
- Aplica una funcion a cada elemento
- **Convierte valores numericos en categorias (binning)** ✅

7️⃣ Que hace apply() en pandas?

- Une dos DataFrames
- Filtra filas segun una condicion
- Ordena el DataFrame
- **Aplica una funcion a cada elemento de una columna** ✅

8️⃣ Que hace pd.concat()?

- Une tablas por columnas en comun
- Aplica una funcion a cada fila
- Crea una tabla resumen
- **Apila DataFrames uno encima del otro** ✅
